In [ ]:
from pyspark.sql import DataFrame, functions as F

from transforms.elo import build_elo

SILVER_TABLE = "stryktipset.silver.matches"
GOLD_TABLE = "stryktipset.gold.fact_match"  # built by 04_gold_fact_match

In [ ]:
def load_matches(table: str = SILVER_TABLE) -> DataFrame:
    """Every match in silver -- all leagues, all countries, played and not.

    Deliberately unscoped. gold's star schema stays England/Sweden, but the
    ratings pool does not: a Stryktipset coupon regularly carries Eliteserien,
    Scottish Premiership or La Liga fixtures (about 16% of matches, ~2 per
    coupon), and those teams need ratings for 07 to be able to score them.
    """
    return spark.table(table)

In [ ]:
def merge_into_fact_match(df_elo: DataFrame, table: str = GOLD_TABLE) -> None:
    """Merge elo_home/elo_away onto the existing fact_match rows, on
    match_key.

    Only in-scope matches exist in fact_match, so the out-of-scope rows in
    df_elo simply find no match and are ignored -- they are still used by 07,
    which reads silver rather than gold.

    Named column assignment, same as 05. The columns are pre-created as nulls
    by 04 because Delta resolves a named assignment before any schema
    evolution runs -- without the placeholders this fails with
    DELTA_MERGE_UNRESOLVED_EXPRESSION.
    """
    source_view = "elo_source"
    df_elo.select("match_id", "elo_home", "elo_away").createOrReplaceTempView(source_view)

    spark.sql(f"""
        MERGE WITH SCHEMA EVOLUTION INTO {table} AS target
        USING {source_view} AS source
        ON target.match_key = source.match_id
        WHEN MATCHED THEN UPDATE SET
            target.elo_home = source.elo_home,
            target.elo_away = source.elo_away
    """)

In [ ]:
def summarise(df_elo: DataFrame, matches: DataFrame) -> str:
    """Print what no unit test can check -- that the ratings are the right
    shape, and that the strongest clubs come out on top -- and return a
    one-line version of it.

    main() hands that line to dbutils.notebook.exit, which makes it the
    task's run output, so a run can be audited from the Jobs API without
    opening the notebook.

    Elo is zero-sum, so the mean should sit near INITIAL_RATING. A mean that
    has drifted far means ratings are leaking somewhere.
    """
    stats = df_elo.select(
        F.count("*").alias("matches"),
        F.round(F.min("elo_home"), 1).alias("min"),
        F.round(F.max("elo_home"), 1).alias("max"),
        F.round(F.avg("elo_home"), 1).alias("mean"),
    ).first()

    top = (
        df_elo.join(matches, on="match_id", how="inner")
        .select("home_team", F.round("elo_home", 1).alias("elo"))
        .orderBy(F.col("elo").desc())
        .limit(10)
        .collect()
    )

    print(f"rated matches : {stats['matches']}")
    print(f"elo_home range: {stats['min']} .. {stats['max']}  (mean {stats['mean']})")
    print()
    print("highest pre-match home ratings on record:")
    for row in top:
        print(f"  {row['home_team']:<30}{row['elo']}")

    listed = ", ".join(f"{row['home_team']} {row['elo']}" for row in top)
    return (
        f"matches={stats['matches']} min={stats['min']} max={stats['max']} "
        f"mean={stats['mean']} | top: {listed}"
    )

In [ ]:
def main():
    """Rate every match in silver, oldest first, and merge the pre-kick-off
    ratings onto the fact_match rows that exist.

    Recomputed in full every run. Elo is sequential -- each match depends on
    every match before it -- so there is no incremental shortcut that stays
    correct once history is backfilled or a result is corrected.
    """
    matches = load_matches()

    df_elo = build_elo(matches)

    summary = summarise(df_elo, matches)

    merge_into_fact_match(df_elo)
    print(f"Rated {df_elo.count()} matches; merged into {GOLD_TABLE}")

    dbutils.notebook.exit(summary)

main()